In [9]:
import pandas as pd

df = pd.read_parquet('../data/reddit_wsb_cache.parquet')

df.head()

,eastern,text,mentioned_tickers
1,2021-01-28 06:32:10-05:00,Math Professor Scott Steiner says the numbers ...,[GME]
2,2021-01-28 06:30:35-05:00,Exit the system The CEO of NASDAQ pushed to ha...,"[GME, QQQ]"
3,2021-01-28 06:28:57-05:00,NEW SEC FILING FOR GME! CAN SOMEONE LESS RETAR...,[GME]
4,2021-01-28 06:26:56-05:00,"Not to distract from GME, just thought our AMC...","[GME, AMC]"
6,2021-01-28 06:26:27-05:00,SHORT STOCK DOESN'T HAVE AN EXPIRATION DATE He...,[T]


In [10]:
unique_tickers = sorted(set().union(*df['mentioned_tickers']))

In [11]:
import yfinance as yf
from concurrent.futures import ThreadPoolExecutor

results = []
maybe_bad_tickers = []

def get_earnings(ticker):
  td = yf.Ticker(ticker)
  actions = td.actions
  if actions is None:
    maybe_bad_tickers.append(ticker)
    return
  earnings = td.get_earnings_dates()
  if earnings is None:
    maybe_bad_tickers.append(ticker)
    return
  actions_dates = pd.to_datetime(actions.index).date
  earnings_dates = pd.to_datetime(earnings.index).date
  all_dates = pd.Series(list(actions_dates) + list(earnings_dates)).drop_duplicates().tolist()
  results.append({'ticker': ticker, 'ignoredates': all_dates})

with ThreadPoolExecutor(max_workers=8) as executor:
  executor.map(get_earnings, unique_tickers)

df = pd.DataFrame(results)

$ABMD: possibly delisted; no timezone found
$ABC: possibly delisted; no timezone found
ABMD: No earnings dates found, symbol may be delisted
ABC: No earnings dates found, symbol may be delisted
$ANSS: possibly delisted; no timezone found
$ATVI: possibly delisted; no timezone found
ATVI: No earnings dates found, symbol may be delisted


KeyboardInterrupt: 

In [ ]:
df.to_parquet('../data/stock_ignoredates.parquet', engine='pyarrow', compression='gzip')

In [ ]:
reddit_posts = pd.read_parquet('../data/reddit_wsb_cache.parquet')
stock_ignore_dates = pd.read_parquet('../data/stock_ignoredates.parquet')
unique_tickers = sorted(set().union(*stock_ignore_dates['ticker']))
stock_prices = []

def get_relevant_prices(ticker):
  
